In [1]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEndpoint , ChatHuggingFace

In [2]:
# Load the doc
from langchain_community.document_loaders import PyPDFLoader
documents = PyPDFLoader("Data/rag_paper.pdf").load()

In [3]:
# split the documents into chunks
from langchain_text_splitters  import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)
chunks = splitter.split_documents(documents = documents)

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# setup the vector store(Dense)
vectorstore = Chroma.from_documents(
    documents = chunks , embedding = embedding
)

In [6]:
# get the dense retriever(semantic retriever)
vector_retriever = vectorstore.as_retriever(
    search_kwagrs = {"k" : 5}
)

In [7]:
query = "Why we need RAG?"
embedd = vector_retriever.invoke(query)

for doc in embedd:
    print(doc.page_content)
    print("-------------------------------------------------------------")

and TriviaQA. RAG compares favourably to the DPR QA system, which uses a BERT-based “cross-
encoder” to re-rank documents, along with an extractive reader. RAG demonstrates that neither a
re-ranker nor extractive reader is necessary for state-of-the-art performance.
There are several advantages to generating answers even when it is possible to extract them. Docu-
ments with clues about the answer but do not contain the answer verbatim can still contribute towards
-------------------------------------------------------------
Broader Impact
This work offers several positive societal beneﬁts over previous work: the fact that it is more
strongly grounded in real factual knowledge (in this case Wikipedia) makes it “hallucinate” less
with generations that are more factual, and offers more control and interpretability. RAG could be
employed in a wide variety of scenarios with direct beneﬁt to society, for example by endowing it
-------------------------------------------------------------
sou

In [8]:
# get the sparse retriever
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

In [9]:
embedd = bm25_retriever.invoke(query)
for doc in embedd:
    print(doc.page_content)
    print("-------------------------------------------------------------")

output sequences,|Y| can become large, requiring many forward passes. For more efﬁcient decoding,
we can make a further approximation thatpθ(y|x,zi)≈ 0 wherey was not generated during beam
search fromx,zi. This avoids the need to run additional forward passes once the candidate set Y has
been generated. We refer to this decoding procedure as “Fast Decoding.”
3 Experiments
We experiment with RAG in a wide range of knowledge-intensive tasks. For all experiments, we use
-------------------------------------------------------------
improves results on all other tasks, especially for Open-Domain QA, where it is crucial.
Index hot-swapping An advantage of non-parametric memory models like RAG is that knowledge
can be easily updated at test time. Parametric-only models like T5 or BART need further training to
update their behavior as the world changes. To demonstrate, we build an index using the DrQA [5]
Wikipedia dump from December 2016 and compare outputs from RAG using this index to the ne

In [10]:
# create the hybrid retriever
hybrid_retriever = EnsembleRetriever(
    retrievers = [vector_retriever , bm25_retriever],
    weights = [0.5 , 0.5]
)

In [11]:
# so now we want to re-ranker
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank

compressor = FlashrankRerank() # we can also use cohere, but flash is fast.

# wrap the hybrid retriever with the reranker
rerank_retriever = ContextualCompressionRetriever(
    base_compressor = compressor,
    base_retriever = hybrid_retriever
)

In [20]:
from langchain_ollama import ChatOllama

# Get the ollama chat model
llm = ChatOllama(
    model = "smollm2:1.7b",
    temperature = 0.1
)

In [21]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system" , "Answer the user's question using ONLY the context provided: \n\n{context}"),
    ("human", "{input}")
])

In [22]:
# now we have multiple chunks of documents from retriever, so we have to merge/stuff
# them into a single context so that we can pass this to the llm

# This chain takes your list of documents, converts them into a single string, 
# and inserts that string into a placeholder (usually called {context}) within prompt.

from langchain_classic.chains.combine_documents import create_stuff_documents_chain

combine_docs_chain = create_stuff_documents_chain(
    llm = llm , prompt = prompt
)

In [23]:
#  now create the final rag chain
from langchain_classic.chains.retrieval import create_retrieval_chain
rag_chain = create_retrieval_chain(
    retriever = hybrid_retriever,
    combine_docs_chain = combine_docs_chain
)

In [24]:
response = rag_chain.invoke({
    "input": "Why we need RAG?"
})

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [25]:
print(response['answer'])

RAG is needed because it combines the strengths of both parametric and non-parametric memory models in a single framework, offering several advantages over traditional seq2seq models:

1. **Improved recall**: Non-parametric memory models can learn to generate answers even when they are not explicitly present in the training data or documents. This improves recall and reduces false negatives.
2. **Better handling of incomplete knowledge**: RAG can handle situations where some information is missing, allowing it to provide more accurate answers than traditional seq2seq models that rely on exact matches.
3. **Flexibility and scalability**: RAG can be easily adapted to different tasks and domains by modifying the memory model and retrieval function. This makes it a versatile tool for various knowledge-intensive NLP applications.
4. **Improved interpretability**: By using non-parametric memory models, RAG provides more transparent and explainable results compared to traditional seq2seq mode